In [1]:
import sys
sys.path.append("/home/landelle/pytorch-cifar/models")
#import util
import resnet
import numpy as np
import torch
import torchvision

import nmixup

In [91]:

transform = torchvision.transforms.transforms.Compose([
    torchvision.transforms.transforms.ToTensor()
])

trainset = torchvision.datasets.CIFAR10(root='/data/landelle/datasets', train=True, download=True, transform=transform)
testset = torchvision.datasets.CIFAR10(root='/data/landelle/datasets', train=False, download=True, transform=transform)

trainset, testset

Files already downloaded and verified
Files already downloaded and verified


(Dataset CIFAR10
     Number of datapoints: 50000
     Root location: /data/landelle/datasets
     Split: Train
     StandardTransform
 Transform: Compose(
                ToTensor()
            ), Dataset CIFAR10
     Number of datapoints: 10000
     Root location: /data/landelle/datasets
     Split: Test
     StandardTransform
 Transform: Compose(
                ToTensor()
            ))

In [3]:

BATCH_SIZE = 100
N_BATCHES_IN_TRAIN_SET = len(trainset) // BATCH_SIZE
N_BATCHES_IN_TEST_SET = len(testset) // BATCH_SIZE

NUM_WORKERS = 8

# Fixed learning rate
LR = 0.05 #0.005 LR=0.1 is used by kuangliu with his implementation of ResNet18 for epochs [0;150)

# CIFAR10 number of classes
NUM_CLASS = 10

# CIFAR10 image metadata
CHANNEL, IMAGE_SIZE, _ = trainset[0][0].shape
print("images are:", IMAGE_SIZE, CHANNEL)


images are: 32 3


In [92]:

trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


In [5]:
train_arr = next(iter(trainloader))[1].numpy()
train_arr.shape

(100,)

In [6]:
def get_n_intraclass_indexing(labels, target_label, n=2):
    pass

def buckets_interclass(labels, n_collectors, n_picks, *, verbose=False):
    """ Returns n collectors of size n_picks randomly picking from disctinct label buckets """
    print_ = lambda *a, **kwa: print(*a, **kwa) if verbose else None
    
    # C is the array of existing classes
    C = np.unique(labels)
    print_("There exists", len(C), "classes.")
    
    # To keep distinct labels at each collector position, we need n_collectors <= C
    assert n_collectors <= len(C)
    
    # Creates C buckets collecting indices of labels of each class, shuffle
    buckets = np.array([np.argwhere(labels == i).T[0] for i in C])
    np.random.shuffle(buckets.T)
    
    # Creates n_collectors collectors that will be returned of size n_picks
    collectors = np.zeros((n_collectors, n_picks), dtype=int)
    
    for pick_i in range(n_picks):
        # Create the array storing remaining buckets indices to choose
        remaining_buckets = np.arange(buckets.shape[0])
        for collector_i in range(n_collectors):
            # Get random bucket index
            bucket_i = np.random.choice(remaining_buckets)
            
            # Forbid picking this bucket again during this pick (no class collision)
            remaining_buckets = remaining_buckets[remaining_buckets != bucket_i]
            
            # Pick randomly chosen index from new randomly chosen bucket
            choice = np.random.choice(buckets[bucket_i])
            
            collectors[collector_i, pick_i] = choice
    
    return collectors




In [7]:
N = 4

labels = np.random.randint(0, N, 1000)
indices = buckets_interclass(labels, N, 100, verbose=True)

# Checks for any class collision in distinct collectors
print([any(labels[indices[i]] == labels[indices[j]]) for i in range(N) for j in range(N) if i < j])

# Checks for any items collision in disctinct collectors
print([any(indices[i] == indices[j]) for i in range(N) for j in range(N) if i < j])


There exists 4 classes.
[False, False, False, False, False, False]
[False, False, False, False, False, False]


In [8]:

def buckets_intraclass(labels, n_collectors, n_picks, *, verbose=False):
    """ Returns n collectors of size n_picks randomly picking from a single label bucket """
    print_ = lambda *a, **kwa: print(*a, **kwa) if verbose else None
    
    # C is the array of existing classes
    C, counts = np.unique(labels, return_counts=True)
    print_("There exists", len(C), "classes. The smallest class is", counts.min(), "long.")
        
    # Our smallest class bucket must be at least n_collectors large to get distinct picks if chosen
    assert n_collectors <= counts.min(),\
    f"Ensure that the smallest class is larger than {n_collectors} items (currently:{counts.min()})"
    
    # Creates C buckets collecting indices of labels of each class, shuffle
    buckets = np.array([np.argwhere(labels == i).T[0] for i in C])
    np.random.shuffle(buckets.T)
    
    # Creates n_collectors collectors that will be returned of size n_picks
    collectors = np.zeros((n_collectors, n_picks), dtype=int)
    
    # Picks a single class bucket to always pick from    
    bucket_i = np.random.randint(buckets.shape[0])
    
    for pick_i in range(n_picks):
        # Create the array storing remaining items indices to choose
        remaining_choices = buckets[bucket_i].copy()
        choice = None
        
        for collector_i in range(n_collectors):
            
            # Pick an item from chosen bucket
            choice = np.random.choice(remaining_choices)
            
            # Remove the item for the rest of the choices
            remaining_choices = remaining_choices[remaining_choices != choice]            
            
            collectors[collector_i, pick_i] = choice
    
    return collectors

In [9]:
N = 5

#labels = np.random.randint(0, N, 20)
labels = np.concatenate([np.arange(N) for _ in range(1000)]) 
np.random.shuffle(labels)

indices = buckets_intraclass(labels, N, 20, verbose=True)

# Checks for any items collision in disctinct collectors
print([any(indices[i] == indices[j]) for i in range(N) for j in range(N) if i < j])



There exists 5 classes. The smallest class is 1000 long.
[False, False, False, False, False, False, False, False, False, False]


In [23]:

trainloader2 = torch.utils.data.DataLoader(trainset, batch_size=len(trainset))
# test_loader = DataLoader(test_dataset, batch_size=len(test_dataset))
X_img = next(iter(trainloader2))[0].numpy()
X_lbl = next(iter(trainloader2))[1].numpy()

#trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
#testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
N = 3
for batch_id, (images, labels) in enumerate(trainloader2):
    print(labels)
    perms = buckets_intraclass(labels, N, 100)

tensor([6, 9, 9,  ..., 9, 1, 1])


ValueError: 'a' must be greater than 0 unless no samples are taken

In [25]:
X_lbl

array([6, 9, 9, ..., 9, 1, 1])

In [29]:
perms = buckets_intraclass(X_lbl, N, 50000)

In [30]:
perms.shape

(3, 50000)

In [36]:
X_lbl[perms[0][0]]

8

In [55]:
X_lbl[perms[2][1]]

8

In [46]:
unique, counts = np.unique(perms[0], return_counts=True)

In [56]:
counts.mean()

10.002000400080016

In [57]:
unique.shape

(4999,)

In [58]:
perms[0]

array([47744,   410, 13460, ..., 20726,  7131, 38599])

In [ ]:
N = 3
for batch_id, (images, labels) in enumerate(trainloader2):
    print(labels)
    perms = buckets_intraclass(labels, N, 100)

In [89]:
perms.shape


(3, 50)

In [69]:
lambda_ = np.array([.5, .5])

In [74]:

USE_CUDA = True
model_getter = nmixup.resnet18

device = torch.device('cuda' if USE_CUDA else 'cpu')
model = model_getter().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = torch.nn.CrossEntropyLoss()
        
lam_rs = lambda_.reshape((-1, 1, 1, 1, 1))
lam_rs_tensor = torch.from_numpy(lam_rs).float().to(device)
        
n_epochs = 10
n_batches = 10
    

In [81]:
def mixup_data_nmix(n_ways, X, y, lam_, perms=None):
    """
    Mixes data n ways with itself shuffled without overlaps, returns X_mixed and an array of every y shuffling
    Example:
        mixup_data(3, X, y, (.5, .5, 0)) will return X_mixed, y[permutation0], y[permutation1], y[permutation2]
        and X_mixed will veriy X_mixed = .5*X[permutation0] + .5*X[permutation1] + 0*X[permutation2]
    X: tensor of shape (batch_size, w, h, 3)
    y: tensor of shape (batch_size,)
    lam_: tensor of shape (n_ways, 1, 1, 1, 1)
    """
    assert lam_.shape[0] == n_ways
    assert X.shape[0] == y.shape[0]
    assert torch.isclose(lam_.sum(), torch.tensor([1]))
    if not type(perms) == np.ndarray:
        perms = no_overlap_perms_random(n_ways, X.shape[0])
    X_permutations = torch.stack([X[perm] for perm in perms], 0)
    X_mixed = (X_permutations * lam_).sum(axis=0)
    ys = [y[perm] for perm in perms]
    return X_mixed, ys, perms

In [96]:


NMIX = lam_rs_tensor.shape[0]
print("Performing n-mixup with N-mix:", NMIX)

# switch to train mode
model.train()
n_epochs = n_epochs if n_epochs else TRAIN_EPOCHS
for epoch in range(1, n_epochs + 1):
    print("Epoch[{}/{}]".format(epoch, n_epochs))
    for batch_id, (images, labels) in enumerate(trainloader):
        print(images.shape, labels.shape)
        if n_batches and batch_id > n_batches:
            break
        labels, images = labels.to(device), images.to(device)
        
        print(images.shape)
        batch_perms = perms[:, 0:50].shape
        print(batch_perms)
        
        
        images, ys, perms = mixup_data_nmix(NMIX, images, labels, lam_rs_tensor, perms)
        #outputs = model(images.float())
        outputs = model(images)
        loss = mixup_criterion_nmix(criterion, outputs, ys, lam_vec)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch_id % 50 == 0:
            print('Loss :{:.4f} Epoch[{}/{}] Batch[{}/{}] batch_shape:{}'.format(
                loss.item(), epoch, n_epochs, batch_id, N_BATCHES_IN_TRAIN_SET, images.shape))

Performing n-mixup with N-mix: 2
Epoch[1/10]
torch.Size([100, 3, 32, 32]) torch.Size([100])


RuntimeError: CUDA error: device-side assert triggered